In [90]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [91]:
spark = SparkSession.builder.appName("NYC").getOrCreate()
base_path = "/home/jovyan/work"

In [92]:
# df = spark.read.parquet(f"{base_path}/vehicle_collisions_partitioned/year=2020/month=1/4f8b4c22c32d4128aac723084b6766f1-0.parquet")
df = spark.read.parquet(f"{base_path}/vehicle_collisions_partitioned/")

df_filtered = df.filter((df.year == 2020) & (df.month == 1))

In [93]:
#TODO: change filter to not only one month but entire year or more

In [94]:
df.printSchema()

root
 |-- CRASH DATE: string (nullable = true)
 |-- CRASH TIME: string (nullable = true)
 |-- BOROUGH: string (nullable = true)
 |-- ZIP CODE: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- LOCATION: string (nullable = true)
 |-- ON STREET NAME: string (nullable = true)
 |-- CROSS STREET NAME: string (nullable = true)
 |-- OFF STREET NAME: string (nullable = true)
 |-- NUMBER OF PERSONS INJURED: double (nullable = true)
 |-- NUMBER OF PERSONS KILLED: double (nullable = true)
 |-- NUMBER OF PEDESTRIANS INJURED: long (nullable = true)
 |-- NUMBER OF PEDESTRIANS KILLED: long (nullable = true)
 |-- NUMBER OF CYCLIST INJURED: long (nullable = true)
 |-- NUMBER OF CYCLIST KILLED: long (nullable = true)
 |-- NUMBER OF MOTORIST INJURED: long (nullable = true)
 |-- NUMBER OF MOTORIST KILLED: long (nullable = true)
 |-- CONTRIBUTING FACTOR VEHICLE 1: string (nullable = true)
 |-- CONTRIBUTING FACTOR VEHICLE 2: string (nullable = tru

In [95]:
df_filtered.show(3, vertical=True)

-RECORD 0---------------------------------------------
 CRASH DATE                    | 01/21/2020           
 CRASH TIME                    | 15:49                
 BOROUGH                       |                      
 ZIP CODE                      |                      
 LATITUDE                      | NULL                 
 LONGITUDE                     | NULL                 
 LOCATION                      |                      
 ON STREET NAME                | BRUCKNER BLVD        
 CROSS STREET NAME             | �ST 138 STREET      
 OFF STREET NAME               |                      
 NUMBER OF PERSONS INJURED     | 0.0                  
 NUMBER OF PERSONS KILLED      | 0.0                  
 NUMBER OF PEDESTRIANS INJURED | 0                    
 NUMBER OF PEDESTRIANS KILLED  | 0                    
 NUMBER OF CYCLIST INJURED     | 0                    
 NUMBER OF CYCLIST KILLED      | 0                    
 NUMBER OF MOTORIST INJURED    | 0                    
 NUMBER OF

In [96]:
# --- remove special chars (ASCII) like e.g. � ---
# --- drop all cols with no lat and long, afterwards drop lat&long - as location has both stored already ---
# --- drop zip code as all other datasets have borough ---

df_filtered = df_filtered.withColumns({
    'ON STREET NAME': regexp_replace('ON STREET NAME', '[^\x00-\x7F]', ''),
    'CROSS STREET NAME': regexp_replace('CROSS STREET NAME', '[^\x00-\x7F]', ''),
    'OFF STREET NAME': regexp_replace('OFF STREET NAME', '[^\x00-\x7F]', '')
})

df_filtered = df_filtered.filter("LATITUDE IS NOT NULL OR LONGITUDE IS NOT NULL")
df_filtered = df_filtered.drop("LATITUDE", "LONGITUDE", "ZIP CODE")

df_filtered.show(3, vertical=True)

-RECORD 0---------------------------------------------
 CRASH DATE                    | 01/10/2020           
 CRASH TIME                    | 0:00                 
 BOROUGH                       |                      
 LOCATION                      |     (40.804707, -... 
 ON STREET NAME                | MAJOR DEEGAN EXPR... 
 CROSS STREET NAME             |                      
 OFF STREET NAME               |                      
 NUMBER OF PERSONS INJURED     | 0.0                  
 NUMBER OF PERSONS KILLED      | 0.0                  
 NUMBER OF PEDESTRIANS INJURED | 0                    
 NUMBER OF PEDESTRIANS KILLED  | 0                    
 NUMBER OF CYCLIST INJURED     | 0                    
 NUMBER OF CYCLIST KILLED      | 0                    
 NUMBER OF MOTORIST INJURED    | 0                    
 NUMBER OF MOTORIST KILLED     | 0                    
 CONTRIBUTING FACTOR VEHICLE 1 | Unsafe Lane Changing 
 CONTRIBUTING FACTOR VEHICLE 2 | Unspecified          
 CONTRIBUT

In [97]:
df_filtered.sort("CRASH DATE", ascending=True).show(3, vertical=True)

-RECORD 0---------------------------------------------
 CRASH DATE                    | 01/01/2020           
 CRASH TIME                    | 19:00                
 BOROUGH                       | QUEENS               
 LOCATION                      | (40.725353, -73.8... 
 ON STREET NAME                | MAIN STREET      ... 
 CROSS STREET NAME             | 72 DRIVE             
 OFF STREET NAME               |                      
 NUMBER OF PERSONS INJURED     | 0.0                  
 NUMBER OF PERSONS KILLED      | 0.0                  
 NUMBER OF PEDESTRIANS INJURED | 0                    
 NUMBER OF PEDESTRIANS KILLED  | 0                    
 NUMBER OF CYCLIST INJURED     | 0                    
 NUMBER OF CYCLIST KILLED      | 0                    
 NUMBER OF MOTORIST INJURED    | 0                    
 NUMBER OF MOTORIST KILLED     | 0                    
 CONTRIBUTING FACTOR VEHICLE 1 | Unspecified          
 CONTRIBUTING FACTOR VEHICLE 2 |                      
 CONTRIBUT

In [98]:
spark.stop()